# Breast Cancer Veri Seti ile Başlangıç

Bu notebook, ödevin ilk aşaması için başlangıç seviyesinde hazırlandı. Amacımız veri setini yüklemek, tabloyu
görmek, kolonları incelemek ve temel yapısını anlamak.


<!-- NOTEBOOK_CSS_STIL_V1: Kod hücrelerini çıktıda daha belirgin yapmak için -->
<style>
/* HTML/PDF (tarayıcıdan yazdır) çıktısında kod alanlarını belirginleştir */

/* Genişlik ve yazdırma uyumu */
@media print {
  body { -webkit-print-color-adjust: exact; print-color-adjust: exact; }
}

/* nbconvert (classic) */
div.input_area {
  background: #f6f8fa;
  border: 1px solid #d0d7de;
  border-left: 8px solid #0b7285;
  border-radius: 10px;
  padding: 12px 14px;
}

div.output_area pre {
  background: #ffffff;
  border: 1px solid #d0d7de;
  border-left: 8px solid #2f9e44;
  border-radius: 10px;
  padding: 10px 14px;
}

/* JupyterLab HTML export'larda görülebilen sınıflar */
div.jp-InputArea-editor, div.jp-RenderedHTMLCommon pre {
  background: #f6f8fa;
  border: 1px solid #d0d7de;
  border-left: 8px solid #0b7285;
  border-radius: 10px;
  padding: 12px 14px;
}

/* Kod fontunu biraz daha okunur yap */
pre, code {
  font-size: 13.5px;
  line-height: 1.35;
}

/* In [ ]: göstergesini daha görünür yap (template'e göre değişebilir) */
div.prompt {
  color: #0b7285;
  font-weight: 700;
}
</style>


## 1. Veri Setinin Yüklenmesi, `X` ve `y` Oluşturma

Bu adımda scikit-learn içindeki **Breast Cancer** veri setini yüklüyorum. Ardından, modeli kurarken kullanacağımız 
özellik matrisi `X` ile hedef değişken `y`’yi oluşturup pandas yapısına (DataFrame/Series) dönüştürüyorum. 
Amaç; veri setinin biçimini, boyutunu ve ilk gözlemlerini netleştirmektir.


### 1.1 Veri Setini Yükleme

Önce gerekli kütüphaneleri içe aktarıyorum ve veri setini yüklüyorum. Ayrıca veri setinin hangi bileşenlerden 
oluştuğunu görmek için `type` ve `keys` bilgilerini inceliyorum.


In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer

dataset = load_breast_cancer()

print('Veri setinin tipi:', type(dataset))
print('Veri setindeki anahtarlar:', list(dataset.keys()))


**Yorum:** `dataset` nesnesi scikit-learn’in veri setlerini taşıyan bir yapıdır. `keys()` çıktısı; ham veri matrisi, 
hedef etiketleri, özellik isimleri ve açıklama metni gibi alanların bulunduğunu gösterir. Bu, veri setini doğru 
şekilde ayrıştırabilmemiz için kritik bir kontroldür.


### 1.2 Ham Veriden İlk Örnekler

Ham veri matrisi (NumPy dizisi) üzerinden ilk 5 satırı inceleyerek değerlerin sayısal yapıda ve çok boyutlu bir 
özellik uzayında olduğunu gözlemliyorum.


In [ ]:
print('Ham veri matrisinden ilk 5 satır:')
print(dataset.data[:5])


**Yorum:** Buradaki çıktı, her satırın bir gözlemi; her sütunun ise bir özelliği temsil ettiğini gösterir. Değerlerin 
ölçeklerinin birbirinden farklı olabileceği görülür; bu durum ilerleyen adımlarda ölçeklendirme (scaling) ihtiyacına 
işaret eder.


### 1.3 `X` ve `y` Oluşturma (pandas)

Şimdi ham veri matrisini özellik isimleriyle birlikte pandas DataFrame’e çeviriyorum (`X`). Hedef değişkeni de 
pandas Series olarak oluşturuyorum (`y`). Bu dönüşüm, veri analizi (EDA) ve modelleme adımlarında daha okunabilir 
ve denetlenebilir bir çalışma sağlar.


In [ ]:
X = pd.DataFrame(dataset.data, columns=dataset.feature_names)
y = pd.Series(dataset.target, name='target')

print('X boyutu (satır, sütun):', X.shape)
print('y boyutu:', y.shape)


**Yorum:** `X.shape` çıktısı gözlem sayısı ve özellik sayısını verir. `y` ise her gözlem için hedef etiketi içerir. 
Bu iki yapının satır sayılarının (örnek sayısının) aynı olması, veri ve etiketlerin doğru hizalandığını gösterir.


### 1.4 DataFrame’i Oluşturma ve İlk 5 Satırı Görme

Raporlama ve hızlı kontrol için `X` ile `y`’yi tek bir DataFrame’de birleştirip ilk 5 satırı görüntülüyorum.


In [ ]:
df = X.copy()
df['target'] = y

print('Birleştirilmiş tablo (df) ilk 5 satır:')
display(df.head())


**Yorum:** İlk 5 satırın incelenmesi; sütunların doğru isimlendiğini, hedef değişkenin tabloya eklendiğini ve veri 
tipinin sayısal olduğunu hızlıca doğrulamamızı sağlar. Bu kontrol, sonraki adımlarda yanlış değişken seçimi veya 
hatalı birleştirme gibi problemleri erken yakalamak açısından önemlidir.


### 1.5 Sınıf Dağılımı (Hedef Değişken)

Son olarak hedef değişkenin sınıf dağılımını kontrol ediyorum. Bu kontrol, sınıf dengesizliği olup olmadığını 
anlamaya yardımcı olur ve stratified split ihtiyacını destekler.


In [ ]:
print('Hedef değişken sınıf dağılımı (adet):')
print(y.value_counts())

print('Hedef değişken sınıf dağılımı (oran):')
print(y.value_counts(normalize=True))


**Yorum:** Sınıf dağılımı, veri setindeki 0 ve 1 etiketlerinin oranlarını gösterir. Eğer ciddi bir dengesizlik varsa, 
model değerlendirmesinde sadece accuracy’ye güvenmek yanıltıcı olabilir. Bu ödevde, stratified bölme kullanarak 
train/validation/test setlerinde sınıf oranlarını mümkün olduğunca korumayı hedefleyeceğiz.


### 1.6 Bölümün Toplu Çalışan Kodu

Aşağıdaki blok, bu bölümde yaptığımız işlemlerin tek parça çalıştırılabilir hâlidir.


In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer

dataset = load_breast_cancer()

print('Veri setinin tipi:', type(dataset))
print('Veri setindeki anahtarlar:', list(dataset.keys()))

print('Ham veri matrisinden ilk 5 satır:')
print(dataset.data[:5])

X = pd.DataFrame(dataset.data, columns=dataset.feature_names)
y = pd.Series(dataset.target, name='target')

print('X boyutu (satır, sütun):', X.shape)
print('y boyutu:', y.shape)

df = X.copy()
df['target'] = y

print('Birleştirilmiş tablo (df) ilk 5 satır:')
display(df.head())

print('Hedef değişken sınıf dağılımı (adet):')
print(y.value_counts())

print('Hedef değişken sınıf dağılımı (oran):')
print(y.value_counts(normalize=True))


## 1. Bölüm Sonu Yorumu

Bu bölümde Breast Cancer veri seti yüklendi; özellik matrisi `X` ve hedef değişken `y` pandas yapısına dönüştürüldü. 
İlk 5 satır üzerinden veri bütünlüğü ve kolon yapısı doğrulandı; ayrıca hedef değişkenin sınıf dağılımı incelenerek 
stratified bölme gerekliliği için temel bir kontrol yapılmış oldu.


# 2. Veri Seti Kalite Kontrolleri

Bu bölümde veri kalitesini önce adım adım inceleyip sonra bölüm sonunda kısa bir toplu kontrol kodu ile toparlıyoruz.

## 2.1 Eksik Değer Analizi

In [ ]:
missing_values = X.isnull().sum()
missing_values

In [ ]:
print('Toplam eksik değer sayısı:', X.isnull().sum().sum())

Bu aşamada sütun sütun eksik değer sayılarını gördük. Eğer tüm sütunlarda sonuç `0` ise veri setinde eksik
veri bulunmadığını söyleyebiliriz.


## 2.2 Veri Tiplerini İnceleme

In [ ]:
X.dtypes

In [ ]:
X.dtypes.value_counts()

Bu çıktı bize veri setindeki sütunların hangi veri tipinde olduğunu gösterir. Tüm değişkenlerin sayısal
olması, sonraki analiz adımlarını daha rahat uygulamamızı sağlar.


## 2.3 Aykırı Değer Analizi

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
plt.figure(figsize=(18, 10))
sns.boxplot(data=X)
plt.xticks(rotation=90)
plt.title('Tüm Özellikler İçin Boxplot Grafiği')
plt.show()

Boxplot grafikleri bazı sütunlarda aykırı değer olabilecek gözlemler bulunduğunu göstermektedir. Özellikle
kutu grafiğinin dışında kalan noktalar dikkat çekmektedir. Bu değerler bazı modelleri etkileyebileceği için
sonraki adımlarda ölçeklendirme ve model seçimi önemli olacaktır.


## 2.4 Bölümün Toplu Çalışan Kodu

In [ ]:
missing_values = X.isnull().sum()
print(missing_values)

print('Toplam eksik değer sayısı:', X.isnull().sum().sum())

print(X.dtypes)
print(X.dtypes.value_counts())

## 2. Bölüm Sonu Yorumu

Veri setinde yapılan kalite kontrolleri sonucunda hiçbir sütunda eksik gözlem bulunmadığı görülmüştür. Ayrıca
tüm değişkenlerin sayısal ve `float64` tipinde olduğu belirlenmiştir. Boxplot incelemesi ise bazı
değişkenlerde aykırı değer olabilecek gözlemler bulunduğunu göstermiştir. Bu durum, veri setinin genel olarak
analize uygun olduğunu; ancak ölçeklendirme ve modelleme aşamalarında aykırı değer etkisinin dikkate alınması
gerektiğini göstermektedir.


# 3. Keşifsel Veri Analizi (EDA)

Bu bölümde, modeli kurmadan önce veriyi daha iyi tanımak için temel istatistiklere ve değişkenler arası ilişkilere bakıyorum.

## 3.1 Temel İstatistikler

Önce sayısal değişkenlerin özetini (ortalama, standart sapma, çeyreklikler vb.) çıkarıyorum.

In [ ]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

# df, 1. bölümde oluşturulmuştu (X + target).
# Burada EDA için hızlı bir özet tablo alıyorum.
istatistik_ozet = df.drop(columns=['target']).describe().T
istatistik_ozet.head()


In [ ]:
# İstersem tüm tabloyu da görüntüleyebilirim:
istatistik_ozet


In [ ]:
# Target'a göre bazı özelliklerin ortalaması nasıl değişiyor?
df.groupby('target').mean(numeric_only=True).iloc[:, :8]  # ilk 8 özelliği örnek olarak gösteriyorum


### 3.1.1 İstenen İstatistiksel Özellikler Tablosu

Hocanın istediği üzere her bir özellik için aşağıdaki özet istatistikleri tek tabloda hesaplıyorum:
- Mean (Ortalama)
- Median (Medyan)
- Min–Max
- Std (Standart sapma)
- Q1–Q3 (1. ve 3. çeyrek)


In [ ]:
import numpy as np

istatistik_tablo = pd.DataFrame({
    'Mean': X.mean(),
    'Median': X.median(),
    'Min': X.min(),
    'Max': X.max(),
    'Std': X.std(),
    'Q1': X.quantile(0.25),
    'Q3': X.quantile(0.75),
})

# İstenen biçimleri ayrıca tek sütunda da gösterelim
istatistik_tablo['Min-Max'] = istatistik_tablo['Min'].round(4).astype(str) + ' – ' + istatistik_tablo['Max'].round(4).astype(str)
istatistik_tablo['Q1-Q3'] = istatistik_tablo['Q1'].round(4).astype(str) + ' – ' + istatistik_tablo['Q3'].round(4).astype(str)

# Rapor açısından daha okunur bir kolon sırası
istatistik_tablo = istatistik_tablo[['Mean', 'Median', 'Min-Max', 'Std', 'Q1-Q3', 'Min', 'Max', 'Q1', 'Q3']]

display(istatistik_tablo.head(10))
print('Not: Yukarıda örnek olarak ilk 10 özellik gösterildi. Tüm tablo için `istatistik_tablo` değişkenini görüntüleyebilirsiniz.')


**Çıktı Yorumu (Akademik):**
Bu tablo, her bir özelliğin merkezi eğilim (mean/median), yayılım (std), uç değer aralığı (min–max) ve çeyreklik aralığı (Q1–Q3)
bilgilerini birlikte sunar. Özellikle mean ile median arasındaki farklar, dağılımın simetrik olup olmadığına dair ilk ipuçlarını verir.
Min–max aralığının genişliği ve std’nin büyüklüğü ise bazı değişkenlerin ölçek olarak diğerlerinden belirgin biçimde farklı olabileceğini
gösterir; bu bulgu ilerleyen adımlarda ölçeklendirme ihtiyacını destekler.


## 3.2 Korelasyon Matrisi ve Heatmap

Korelasyon, iki değişkenin birlikte nasıl değiştiğini gösterir. Çok yüksek korelasyonlar bazı modellerde
(özellikle doğrusal) benzer bilgiyi tekrar taşıyor olabilir.


In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

# Tüm özellikler için korelasyon matrisi
korelasyon = X.corr()

# Heatmap çok kalabalık olmasın diye, hedef değişkenle en ilişkili ilk 12 özelliği seçiyorum
hedef_korelasyon = X.corrwith(y).abs().sort_values(ascending=False)
secili_ozellikler = hedef_korelasyon.index[:12]
korelasyon_secili = korelasyon.loc[secili_ozellikler, secili_ozellikler]

plt.figure(figsize=(10, 8))
sns.heatmap(korelasyon_secili, cmap='coolwarm', center=0, annot=False)
plt.title('Korelasyon Heatmap (Seçili 12 Özellik)')
plt.tight_layout()

def repo_kokunu_bul() -> Path:
    # Önce Git reposunun kökünü (.git üzerinden) bulmaya çalışıyorum
    cwd = Path.cwd()
    for p in [cwd, *cwd.parents]:
        if (p / '.git').exists():
            return p
    # Colab vb. ortamlarda .git olmayabilir; o zaman mevcut dizini kök kabul ediyorum
    return cwd

repo_kok = repo_kokunu_bul()
figures_dir = repo_kok / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)
cikis_dosyasi = figures_dir / '03_korelasyon_heatmap.svg'

plt.savefig(cikis_dosyasi)
print('Kaydedildi:', cikis_dosyasi.resolve())
plt.show()


In [ ]:
# En yüksek mutlak korelasyona sahip ilk 3 değişken çifti
korelasyon_abs = korelasyon.abs()
ust_ucgen = korelasyon_abs.where(np.triu(np.ones(korelasyon_abs.shape), k=1).astype(bool))
en_yuksek = (
    ust_ucgen.stack()
    .sort_values(ascending=False)
    .head(3)
)
en_yuksek


## 3. Bölüm Sonu Yorumu

Bu adımda verinin genel dağılımını ve değişkenlerin birbiriyle ilişkisini gördüm. Korelasyon matrisi özellikle
çok benzer bilgi taşıyan özellikleri fark etmeme yardımcı oldu.


# 4. Veri Ölçeklendirme (Scaling)

Ne yapıyoruz?
- Özelliklerin (kolonların) ölçeklerini inceliyoruz ve neden ölçeklendirme gerektiğini gerekçelendiriyoruz.
- Ölçeklendirmeyi **split işleminden sonra** (5. adımda) uygulayacağız. Burada amaç, yöntemi doğru yerde kullanmak ve veri sızıntısını önlemektir.


In [ ]:
# Ölçekler hakkında hızlı bir fikir edinmek için bazı özet istatistiklere bakıyorum
olcek_ozeti = X.describe().loc[['mean', 'std', 'min', 'max']].T
display(olcek_ozeti.head(10))

print('Örnek olarak ilk 10 kolon gösterildi. Tüm kolonların ölçeklerini görmek isterseniz `olcek_ozeti` değişkenini görüntüleyebilirsiniz.')


**Çıktı Yorumu (Akademik):**
Özet istatistikler, bazı özelliklerin değer aralıklarının diğerlerine göre çok daha büyük olabildiğini göstermektedir.
Bu durum özellikle doğrusal modellerde ve PCA gibi varyans-temelli yöntemlerde, büyük ölçekli değişkenlerin modele gereğinden
fazla ağırlık vermesine neden olabilir. Bu nedenle StandardScaler gibi bir ölçeklendirme yaklaşımı kullanılacaktır.


In [ ]:
from sklearn.preprocessing import StandardScaler

# Not: scaler'ı burada tanımlıyorum; fit/transform işlemini veri sızıntısını önlemek için 5. adımda split sonrasında yapacağım.
scaler = StandardScaler()
print('StandardScaler hazır. (fit işlemi split sonrası uygulanacak)')


**Çıktı Yorumu (Akademik):**
Ölçeklendirme işleminin doğru uygulanabilmesi için, dönüşüm parametrelerinin yalnızca eğitim verisinden öğrenilmesi gerekir.
Bu yaklaşım, validation ve test setlerinin eğitim sürecine dolaylı olarak dahil olmasını (data leakage) engeller.


# 5. Veri Setinin Bölünmesi

Ne yapıyoruz?
- Veri setini **train / validation / test** olarak 70/10/20 oranında ayırıyoruz.
- Sınıf oranlarını korumak için **stratified split** kullanıyoruz.
- Ardından (4. adımda tanımladığımız) ölçeklendirmeyi sadece `train` üzerinde öğrenip, `validation` ve `test` setlerine uyguluyoruz.


In [ ]:
from sklearn.model_selection import train_test_split

# 1) Önce test setini ayır (20%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 2) Kalan 80% içinden validation ve test'i eşit böl (10% + 10%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print('Eğitim (train) boyutu:', X_train.shape, 'Hedef:', y_train.shape)
print('Doğrulama (validation) boyutu:', X_val.shape, 'Hedef:', y_val.shape)
print('Test boyutu:', X_test.shape, 'Hedef:', y_test.shape)


In [ ]:
print('Train sınıf oranları:')
print(y_train.value_counts(normalize=True))

print('Validation sınıf oranları:')
print(y_val.value_counts(normalize=True))

print('Test sınıf oranları:')
print(y_test.value_counts(normalize=True))


**Çıktı Yorumu (Akademik):**
Stratified bölme sayesinde, train/validation/test alt kümelerinde sınıf oranlarının birbirine yakın kaldığı görülür.
Bu durum, performans karşılaştırmalarının daha adil yapılmasına katkı sağlar.


In [ ]:
import pandas as pd

# Ölçeklendirme: fit sadece train, transform val/test
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

# Kolon isimlerini kaybetmemek için DataFrame'e çeviriyorum
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
X_val_scaled   = pd.DataFrame(X_val_scaled,   columns=X.columns, index=X_val.index)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=X.columns, index=X_test.index)

print('Ölçeklendirilmiş train boyutu:', X_train_scaled.shape)
display(X_train_scaled.head())


**Çıktı Yorumu (Akademik):**
Ölçeklendirme sonrası veri boyutları korunur ve dönüşüm parametreleri yalnızca eğitim verisinden öğrenildiği için veri sızıntısı
riski azaltılmış olur. Bu ölçeklendirilmiş veri, PCA/LDA ve bazı modeller için daha uygun bir girdi sağlar.


# 6. Özellik Seçimi ve Boyut İndirgeme

Ne yapıyoruz?
- PCA ile etiket kullanmadan (unsupervised) boyut indirgeme yapıyoruz.
- LDA ile etiket bilgisi kullanarak (supervised) sınıfları ayıran doğrusal ekseni elde ediyoruz.
- Her iki dönüşümü de yalnızca `train` üzerinde öğrenip, `validation` ve `test` setlerine uyguluyoruz.


In [ ]:
from pathlib import Path

def repo_kokunu_bul() -> Path:
    cwd = Path.cwd()
    for p in [cwd, *cwd.parents]:
        if (p / '.git').exists():
            return p
    return cwd

repo_kok = repo_kokunu_bul()
figures_dir = repo_kok / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)
print('Şekiller klasörü:', figures_dir.resolve())


## 6.1 PCA (Boyut İndirgeme)

Ne yapıyoruz?
- PCA’yı `train` setinde öğreniyoruz.
- Ödev kuralına göre, explained variance oranı ortalamasından büyük olan bileşenleri seçiyoruz.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

pca_tam = PCA(random_state=42)
pca_tam.fit(X_train_scaled)

oranlar = pca_tam.explained_variance_ratio_
esik = oranlar.mean()
secili_bilesen_sayisi = int((oranlar > esik).sum())

print('Toplam bileşen sayısı:', len(oranlar))
print('Ortalama explained variance oranı:', esik)
print('Seçilen bileşen sayısı:', secili_bilesen_sayisi)


In [ ]:
kumulatif = np.cumsum(oranlar)

plt.figure(figsize=(10, 4))
plt.plot(oranlar, marker='o', linewidth=1, label='Explained variance ratio')
plt.plot(kumulatif, marker='.', linewidth=2, label='Kümülatif')
plt.axhline(esik, color='red', linestyle='--', linewidth=1, label='Ortalama eşik')
plt.title('PCA Explained Variance Oranları (Train)')
plt.xlabel('Bileşen indeksi')
plt.ylabel('Oran')
plt.legend()
plt.tight_layout()

cikis = figures_dir / '06_pca_explained_variance.svg'
plt.savefig(cikis)
print('Kaydedildi:', cikis.resolve())
plt.show()


**Çıktı Yorumu (Akademik):**
Seçilen bileşen sayısı, ödevde belirtilen kurala göre varyansı ortalamanın üzerinde açıklayan bileşenlerin sayısıdır.
Kümülatif varyans eğrisi ise seçilen bileşenlerin toplam bilginin ne kadarını taşıdığını gösterir.


In [ ]:
pca = PCA(n_components=secili_bilesen_sayisi, random_state=42)

X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca   = pca.transform(X_val_scaled)
X_test_pca  = pca.transform(X_test_scaled)

print('Train PCA boyutu:', X_train_pca.shape)
print('Validation PCA boyutu:', X_val_pca.shape)
print('Test PCA boyutu:', X_test_pca.shape)


In [ ]:
import seaborn as sns

pca_viz = PCA(n_components=2, random_state=42)
X_train_2d = pca_viz.fit_transform(X_train_scaled)

pca_df = pd.DataFrame(X_train_2d, columns=['PC1', 'PC2'], index=y_train.index)
pca_df['target'] = y_train.values

plt.figure(figsize=(7, 6))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='target', palette='Set2', alpha=0.7)
plt.title('PCA 2B Scatter (Train)')
plt.tight_layout()

cikis = figures_dir / '06_pca_scatter_train.svg'
plt.savefig(cikis)
print('Kaydedildi:', cikis.resolve())
plt.show()


## 6.2 LDA (Sınıf Ayırıcı Boyut İndirgeme)

Ne yapıyoruz?
- LDA etiketi (`y`) kullanır ve sınıfları ayıran doğrusal ekseni bulur.
- İki sınıflı problemde maksimum bileşen sayısı `n_classes - 1 = 1` olduğu için 1 bileşenle ilerliyoruz.


In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

lda = LinearDiscriminantAnalysis(n_components=1)

X_train_lda = lda.fit_transform(X_train_scaled, y_train)
X_val_lda   = lda.transform(X_val_scaled)
X_test_lda  = lda.transform(X_test_scaled)

print('Train LDA boyutu:', X_train_lda.shape)
print('Validation LDA boyutu:', X_val_lda.shape)
print('Test LDA boyutu:', X_test_lda.shape)


In [ ]:
lda_df = pd.DataFrame({'LD1': X_train_lda.ravel(), 'target': y_train.values})

plt.figure(figsize=(8, 4))
sns.histplot(data=lda_df, x='LD1', hue='target', stat='density', common_norm=False, bins=30, element='step')
plt.title('LDA Dağılımı (Train)')
plt.tight_layout()

cikis = figures_dir / '06_lda_hist_train.svg'
plt.savefig(cikis)
print('Kaydedildi:', cikis.resolve())
plt.show()


## 6. Bölüm Sonu Yorumu

Bu adımda PCA ve LDA ile iki farklı boyut indirgeme temsili elde edildi. Bir sonraki aşamada ham (ölçeklendirilmiş), PCA ve LDA
temsilini kullanarak modelleri kuracak ve validation performanslarını karşılaştıracağız.


# 7. Makine Öğrenmesi Modellerinin Kurulması

Ne yapıyoruz?
- Ödevde istenen 5 farklı modeli tanımlıyoruz: Logistic Regression, Decision Tree, Random Forest, XGBoost, GaussianNB.
- Bu modelleri 3 farklı temsil üzerinde (ham/ PCA/ LDA) aynı şekilde eğitmeye hazır bir yapı kuruyoruz.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

try:
    from xgboost import XGBClassifier
    xgb_var = True
except Exception as e:
    xgb_var = False
    print('Uyarı: xgboost yüklenemedi. requirements.txt kurulumunu kontrol edin. Hata:', e)

modeller = {
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42),
    'GaussianNB': GaussianNB(),
}

if xgb_var:
    modeller['XGBoost'] = XGBClassifier(
        n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9,
        reg_lambda=1.0, random_state=42, eval_metric='logloss'
    )

print('Tanımlanan modeller:', list(modeller.keys()))


**Çıktı Yorumu (Akademik):**
Bu aşamada modeller sadece tanımlanmıştır. Eğitim ve performans ölçümü bir sonraki adımda (validation değerlendirmesi) yapılacaktır.


# 8. Validation Performanslarının Ölçülmesi

Ne yapıyoruz?
- Her model için validation seti üzerinde metrikleri (Accuracy, Precision, Recall, F1-score, ROC-AUC) hesaplıyoruz.
- Ham (ölçeklendirilmiş), PCA ve LDA temsilleri için sonuçları tek tabloda karşılaştırıyoruz.
- Eğer sonuçlar olağanüstü iyi (ör. 1.0) görünüyorsa bunu not edip, çapraz doğrulama gibi ek kontrollerle sorguluyoruz.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def metrikleri_hesapla(y_true, y_pred, y_prob):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
        'roc_auc': roc_auc_score(y_true, y_prob),
    }

veri_temsilleri = {
    'Ham (Ölçeklendirilmiş)': (X_train_scaled, y_train, X_val_scaled, y_val, X_test_scaled, y_test),
    'PCA': (X_train_pca, y_train, X_val_pca, y_val, X_test_pca, y_test),
    'LDA': (X_train_lda, y_train, X_val_lda, y_val, X_test_lda, y_test),
}

sonuclar = []

for temsil_adi, (Xtr, ytr, Xva, yva, Xte, yte) in veri_temsilleri.items():
    for model_adi, model in modeller.items():
        model.fit(Xtr, ytr)
        y_pred = model.predict(Xva)
        if hasattr(model, 'predict_proba'):
            y_prob = model.predict_proba(Xva)[:, 1]
        else:
            # Bu ödevde kullandığımız modellerde predict_proba var; yine de güvenli olsun
            y_prob = y_pred
        m = metrikleri_hesapla(yva, y_pred, y_prob)
        m.update({'temsil': temsil_adi, 'model': model_adi})
        sonuclar.append(m)

sonuclar_df = pd.DataFrame(sonuclar)
sonuclar_df = sonuclar_df.sort_values(by=['roc_auc', 'f1'], ascending=False)
display(sonuclar_df.head(15))

print('Toplam model sayısı:', len(sonuclar_df))


In [ ]:
en_iyi_satir = sonuclar_df.iloc[0]
print('''Validation'a göre en iyi seçim:''')
print('Temsil:', en_iyi_satir['temsil'])
print('Model :', en_iyi_satir['model'])
print('ROC-AUC:', en_iyi_satir['roc_auc'])
print('F1    :', en_iyi_satir['f1'])


**Çıktı Yorumu (Akademik):**
Validation sonuçları, farklı temsil ve modellerin karşılaştırmalı performansını gösterir. Eğer bazı kombinasyonlarda ROC-AUC veya
Accuracy değerleri 1.0’a çok yakın ya da tam 1.0 ise, bu durum veri setinin ayrılabilirliğinden kaynaklanabileceği gibi validation
setinin görece küçük olması nedeniyle de görülebilir. Bu nedenle, en iyi modelin test seti üzerindeki performansı ayrıca raporlanmalıdır.


# 9. En İyi Modelin Test Üzerinde Değerlendirilmesi

Ne yapıyoruz?
- Validation sonuçlarına göre seçilen en iyi model/temsil kombinasyonunu alıyoruz.
- Modeli eğitim + doğrulama verisi üzerinde yeniden eğitip, sadece test seti üzerinde değerlendiriyoruz.
- Confusion matrix ve ROC curve gibi grafiklerle performansı raporluyoruz.


In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    roc_auc_score,
    accuracy_score,
    f1_score
)
import matplotlib.pyplot as plt
import pandas as pd

def temsil_verisini_getir(temsil_adi):
    Xtr, ytr, Xva, yva, Xte, yte = veri_temsilleri[temsil_adi]
    return Xtr, ytr, Xva, yva, Xte, yte

secili_temsil = en_iyi_satir['temsil']
secili_model_adi = en_iyi_satir['model']

Xtr, ytr, Xva, yva, Xte, yte = temsil_verisini_getir(secili_temsil)

# Train + Validation setlerini birleştirerek final modeli yeniden eğit
# DataFrame yapısını korumak için pd.concat kullanıyoruz
X_trainval = pd.concat([Xtr, Xva], axis=0)
y_trainval = pd.concat([ytr, yva], axis=0)

# Final modeli seç ve eğit
model_final = modeller[secili_model_adi]
model_final.fit(X_trainval, y_trainval)

# Test tahminleri
y_test_pred = model_final.predict(Xte)

# Olasılık tahmini varsa ROC-AUC için kullan
if hasattr(model_final, 'predict_proba'):
    y_test_prob = model_final.predict_proba(Xte)[:, 1]
else:
    y_test_prob = y_test_pred

# Test metrikleri
print('Test ROC-AUC:', roc_auc_score(yte, y_test_prob))
print('Test Accuracy:', accuracy_score(yte, y_test_pred))
print('Test F1-score:', f1_score(yte, y_test_pred))

# Confusion Matrix
cm = confusion_matrix(yte, y_test_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues')
plt.title(f'Confusion Matrix - {secili_model_adi} ({secili_temsil})')
plt.show()

# ROC Curve
RocCurveDisplay.from_predictions(yte, y_test_prob)
plt.title(f'ROC Curve - {secili_model_adi} ({secili_temsil})')
plt.show()


In [ ]:
# Confusion Matrix
cm = confusion_matrix(yte, y_test_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(values_format='d')
plt.title('Confusion Matrix (Test)')
plt.tight_layout()
cikis = figures_dir / '09_confusion_matrix_test.svg'
plt.savefig(cikis)
print('Kaydedildi:', cikis.resolve())
plt.show()


In [ ]:
# ROC Curve
roc_disp = RocCurveDisplay.from_predictions(yte, y_test_prob)
plt.title('ROC Curve (Test)')
plt.tight_layout()
cikis = figures_dir / '09_roc_curve_test.svg'
plt.savefig(cikis)
print('Kaydedildi:', cikis.resolve())
plt.show()


**Çıktı Yorumu (Akademik):**
Test seti, modelin genellenebilirliğini değerlendirmek için ayrılmıştır ve eğitim/validasyon sürecine dahil edilmemelidir.
Confusion matrix, yanlış sınıflandırma sayısını açıkça gösterirken; ROC eğrisi eşik-bağımsız performansı görselleştirir.
Eğer test performansı da olağanüstü yüksek çıkarsa, veri setinin ayrılabilirliği yanında veri sızıntısı olasılığına karşı
kullanılan veri akışı (split → fit scaler/PCA/LDA yalnızca train) tekrar kontrol edilmelidir.


# 10. XAI – SHAP Açıklanabilirlik Analizi (Zorunlu)

Ne yapıyoruz?
- En iyi modelin kararlarını hangi özelliklerin ne yönde etkilediğini SHAP ile açıklıyoruz.
- En azından bir `summary_plot` ve bir önem sıralaması grafiği üreteceğiz.

Not: SHAP analizi model türüne göre farklı açıklayıcılar (TreeExplainer/LinearExplainer) kullanır.


In [ ]:
import numpy as np

try:
    import shap
    shap_var = True
except Exception as e:
    shap_var = False
    print('Uyarı: shap yüklenemedi. requirements.txt kurulumunu kontrol edin. Hata:', e)

print('SHAP hazır mı?:', shap_var)


In [ ]:
# SHAP analizi (uygun model türlerinde)
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# SHAP analizi
if not shap_var:
    print('SHAP kurulumu olmadığı için bu adım atlandı.')
else:
    # Temsile göre test verisi ve özellik isimleri
    if secili_temsil == 'Ham (Ölçeklendirilmiş)':
        X_test_for_shap = Xte.copy()
        if not isinstance(X_test_for_shap, pd.DataFrame):
            X_test_for_shap = pd.DataFrame(X_test_for_shap, columns=X.columns)
        feature_names = list(X_test_for_shap.columns)

    elif secili_temsil == 'PCA':
        X_test_for_shap = pd.DataFrame(
            Xte,
            columns=[f'PC{i+1}' for i in range(Xte.shape[1])]
        )
        feature_names = list(X_test_for_shap.columns)

    else:  # LDA
        X_test_for_shap = pd.DataFrame(
            Xte,
            columns=[f'LD{i+1}' for i in range(Xte.shape[1])]
        )
        feature_names = list(X_test_for_shap.columns)

    # Örneklem al
    X_sample = X_test_for_shap.sample(n=min(200, len(X_test_for_shap)), random_state=42)

    model_for_shap = model_final

    try:
        # Genel ve güvenli kullanım
        explainer = shap.Explainer(model_for_shap, X_sample)
        shap_values = explainer(X_sample)

        # Summary plot
        shap.summary_plot(shap_values, X_sample, feature_names=feature_names, show=False)
        cikis = figures_dir / '10_shap_summary.svg'
        plt.tight_layout()
        plt.savefig(cikis)
        print('Kaydedildi:', cikis.resolve())
        plt.show()

        # Bar plot
        shap.plots.bar(shap_values, show=False)
        cikis = figures_dir / '10_shap_bar.svg'
        plt.tight_layout()
        plt.savefig(cikis)
        print('Kaydedildi:', cikis.resolve())
        plt.show()

    except Exception as e:
        print('SHAP bu model/temsil için çalışmadı. Hata:', e)


**Çıktı Yorumu (Akademik):**
SHAP grafikleri, modelin hangi özellikleri daha etkili kullandığını ve bu özelliklerin tahminleri hangi yönde etkilediğini gösterir.
Özellikle `summary_plot`, örnek bazında etki dağılımını verirken; bar grafiği ortalama mutlak SHAP değerleri üzerinden genel önem sıralaması
sunar. Bu yorumlar, modelin sadece performans açısından değil, karar mekanizması açısından da raporlanmasına imkan tanır.
